# Praktikum - Logistic Regression

**Studi Kasus:** Prediksi tingkat stress mahasiswa berdasarkan pola tugas dan jam tidur menggunakan logistic regression.

## 1. Membaca Dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Membaca dataset
df = pd.read_csv('Survei Beban Tugas & Tingkat Stres Mahasiswa  (Jawaban)(Form Responses 1) (1).csv')
display(df.head())

## 2. Data Cleaning

Karena data sebagian besar sudah melalui proses pembersihan, kita hanya perlu menyesuaikan dataset agar siap digunakan untuk *machine learning*:
1. Menghapus kolom yang tidak akan dimasukkan ke dalam model (`Timestamp`, `Stress?`, `Nama Lengkap`, `NIM`).
2. Menghapus jika ada *missing value* sisa (baris kosong).

In [ ]:
# Menghapus kolom yang tidak relevan
# Kita akan langsung memakai kolom 'Stress' yang nilainya sudah 1 dan 0.
df_clean = df.drop(columns=['Timestamp', 'Stress?', 'Nama Lengkap', 'NIM'], errors='ignore')

# Memastikan jika masih ada sisa baris kosong (missing value) bisa dihapus
df_clean = df_clean.dropna()

display(df_clean.info())
display(df_clean.head())

## 3. Feature Selection

Memilih fitur (`jumlah tugas`, `jumlah deadline`, `rata rata jam tidur`) dan target (`Stress`).

In [ ]:
# Pemilihan fitur dan target
X = df_clean[['jumlah tugas', 'jumlah deadline', 'rata rata jam tidur']]
y = df_clean['Stress'].astype(int)

display(X.head())
display(y.head())

## 4. Split Data

Membagi dataset menjadi data training dan data testing.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Jumlah data training: {X_train.shape[0]}")
print(f"Jumlah data testing: {X_test.shape[0]}")

## 5. Training Model

Melatih model menggunakan `LogisticRegression()`.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
print("Model Logistic Regression berhasil dilatih!")

## 6. Prediction

Melakukan prediksi terhadap data testing.

In [ ]:
y_pred = model.predict(X_test)
print("Hasil prediksi pada data testing:")
print(y_pred)

## 7. Evaluation

Menampilkan accuracy, classification report, dan visualisasi confusion matrix menggunakan Seaborn.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Tidak Stress', 'Stress'], yticklabels=['Tidak Stress', 'Stress'])
plt.title('Confusion Matrix')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

## 8. Probabilitas Prediksi

Menampilkan probabilitas hasil prediksi menggunakan `predict_proba()`.

In [ ]:
y_pred_proba = model.predict_proba(X_test)
print("Probabilitas Prediksi (Kelas 0: Tidak Stress, Kelas 1: Stress):\n")
for i, proba in enumerate(y_pred_proba):
    print(f"Data ke-{i+1}: {np.round(proba, 2)} -> Prediksi: {y_pred[i]} (Aktual: {y_test.iloc[i]})")

print("\nContoh Interpretasi Data ke-1:")
print(f"- {np.round(y_pred_proba[0][0]*100, 1)}% kemungkinan tidak stress")
print(f"- {np.round(y_pred_proba[0][1]*100, 1)}% kemungkinan stress")

## Analisis

**1. Mengapa logistic regression cocok untuk kasus ini?**
Logistic regression sangat cocok untuk kasus ini karena tujuannya adalah melakukan klasifikasi biner, yaitu memprediksi salah satu dari dua kemungkinan label: **0 = Tidak Stress** dan **1 = Stress**. Logistic regression sangat efisien dalam memodelkan dan menghitung probabilitas bahwa seorang mahasiswa tergolong kelas "Stress" berdasarkan kombinasi linier dari fitur-fiturnya (tugas, deadline, jam tidur).

**2. Mengapa probabilitas penting dalam prediksi stress?**
Probabilitas sangat penting karena memberikan **tingkat keyakinan (confidence)** atas prediksi, bukan sekadar hasil mutlak klasifikasi. Misalnya, jika probabilitas prediksinya [0.49, 0.51], meskipun model memprediksi mahasiswa tersebut "Stress", ini menunjukkan tingkat ketidakpastian yang tinggi (mahasiswa berada di ambang batas). Hal ini berguna dalam dunia nyata (terutama psikologi dan konseling) karena kita bisa mengambil tindakan preventif/intervensi pada mahasiswa yang sudah menunjukkan tren kemungkinan stress (misal: probabilitas stress di angka 40-50%) sebelum mereka benar-benar stress berat.

**3. Faktor apa yang paling memengaruhi stress mahasiswa?**
Berdasarkan model Logistic Regression, kita dapat melihat dari nilai *koefisien* masing-masing fitur (bisa dilihat pada cell kode di bawah ini). Biasanya, **jumlah tugas** dan **jumlah deadline** memiliki korelasi positif (makin tinggi tugas/deadline, peluang stress makin tinggi). Sedangkan **jam tidur** memiliki korelasi negatif (semakin rendah jam tidur, peluang stress semakin tinggi).

**4. Apa kelemahan model yang dibuat?**
- **Asumsi Linearitas:** Logistic regression berasumsi bahwa terdapat hubungan linier antara *log-odds* (peluang target) dengan variabel prediktor (fitur). Jika pola data mahasiswa lebih kompleks dan non-linier, model ini mungkin kurang optimal.
- **Jumlah Data Sangat Sedikit:** Dataset survei kelas ini cukup kecil (sekitar 40an baris data). Hal ini membuat model berisiko tinggi mengalami *overfitting* atau *underfitting*, dan model tidak bisa dengan mudah digeneralisasi untuk mewakili seluruh mahasiswa di dunia nyata.
- **Sensitif Terhadap Outliers:** Jika ada mahasiswa yang begadang tapi tidak stres, atau sebaliknya banyak tidur tapi stres berat (sebagai anomali), model regresi logistik cukup sensitif dan bisa memengaruhi garis batas keputusannya (*decision boundary*).

In [ ]:
# Melihat Koefisien Model untuk mengetahui faktor yang paling berpengaruh (Analisis No. 3)
coef_df = pd.DataFrame({'Fitur': X.columns, 'Koefisien (Bobot)': model.coef_[0]})
display(coef_df.sort_values(by='Koefisien (Bobot)', ascending=False))